# geo0641 – From Open Data to a Map: DWD Climate Stations Pipeline

**Course:** 9544 Applied Geoinformatics (M.Sc. IECS) · Session 1 · 2026-10-05
**Unit:** geo0640 DWD CDC Stations & Time Series

In this session we build a complete, small geospatial data pipeline:

```
DWD Open Data (HTTPS)  →  raw text  →  pandas DataFrame  →  GeoDataFrame  →  GeoPackage  →  QGIS / web map
        acquire             inspect        parse & clean        georeference      persist        view & check
```

**Learning goals.** After this notebook you can
1. download an open dataset reproducibly via HTTPS and cache it locally,
2. inspect an unknown text format and parse it robustly (fixed-width, encoding, dates),
3. filter tabular data with boolean masks,
4. turn coordinates into geometries with a defined CRS,
5. persist the result as a GeoPackage and inspect it in QGIS on top of an OGC WMS basemap.

> Motto of the course: **Python does the work, PostGIS keeps the data, QGIS lets you see, check and interact.**

## 0. Setup
We use only well-known packages. On the EOLab JupyterHub everything is pre-installed.

In [ ]:
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd

print("pandas", pd.__version__, "| geopandas", gpd.__version__)

In [ ]:
# Where do the data live?
# On the JupyterHub a read-only shared copy may exist; otherwise we download into ./data.
SHARED_DIR = Path("/srv/data/dwd/cdc/annual/kl/historical")
LOCAL_DIR = Path("data/original/dwd/annual/kl/historical")
OUT_DIR = Path("data/derived")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = ("https://opendata.dwd.de/climate_environment/CDC/"
            "observations_germany/climate/annual/kl/historical/")
STATION_FILE = "KL_Jahreswerte_Beschreibung_Stationen.txt"

## 1. Acquire: download once, then work offline
Open the [DWD CDC directory](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/annual/kl/historical/) in your browser.
It is a plain HTML directory listing. The station description file lists all stations of the annual climate (KL) collection.

**Good practice:** download into `data/original/` and never edit original files. Everything we produce goes to `data/derived/`.

In [ ]:
def fetch(url: str, dest: Path, overwrite: bool = False) -> Path:
    """Download url to dest unless a local copy exists. Returns the local path."""
    if dest.exists() and not overwrite:
        print(f"cached: {dest}")
        return dest
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"downloaded {len(r.content):,} bytes -> {dest}")
    return dest


shared = SHARED_DIR / STATION_FILE
station_path = shared if shared.exists() else fetch(BASE_URL + STATION_FILE, LOCAL_DIR / STATION_FILE)
station_path

## 2. Inspect: look at the raw bytes before you parse
**Predict:** what kind of file is this? Which separator? Which encoding? Which problems do you expect?

In [ ]:
raw = station_path.read_bytes()
print(raw[:400])

In [ ]:
for line in raw.decode("latin-1").splitlines()[:6]:
    print(repr(line[:160]))

Observations to discuss:
* **Fixed-width** columns, no separator. Station names contain blanks (`Aachen-Orsbach`, `Bergzabern, Bad`), so `split()` is not safe.
* **Encoding** is ISO-8859-1 (Latin-1): `Württemberg`.
* **Dates** as `YYYYMMDD` strings.
* Line 2 is a row of dashes that *looks* like a column-width specification …

## 3. Parse: a robust reader for the DWD station list
### 3.1 Does the dashed header tell the truth?
Let us measure where the values really start and end in a data line and compare with the dashes.

In [ ]:
import re

lines = raw.decode("latin-1").splitlines()
print("header dashes:")
for m in re.finditer(r"-+", lines[1]):
    print(f"  {m.start():3d}-{m.end():3d}")
print("tokens of a data line:")
for m in re.finditer(r"\S+", lines[3]):
    print(f"  {m.start():3d}-{m.end():3d}  {m.group()}")

The dashes do **not** match the data: e.g. the station id has 11 dashes but 5 digits, and the state column is much wider than its 10 dashes.
Lesson: **check the data, not the documentation.** We define the column positions ourselves.

### 3.2 Exercise: read the file with `pd.read_fwf`
Use the column positions below, the English column names, skip the two header lines, and choose the right encoding.
Keep the dates as strings first (`dtype`), we convert them afterwards.

In [ ]:
NAMES = ["station_id", "date_from", "date_to", "altitude", "lat", "lon", "name", "state", "access"]
COLSPECS = [(0, 5), (6, 14), (15, 23), (24, 38), (39, 50), (51, 60), (61, 102), (102, 143), (143, None)]

# YOUR CODE HERE
df.head()

In [ ]:
# Convert types and use the station id as index
# YOUR CODE HERE
df.dtypes

### 3.3 Quality checks
Never skip this step. How many stations? Missing values? Plausible value ranges?

In [ ]:
print(len(df), "stations")
print(df.isna().sum())
df[["altitude", "lat", "lon"]].describe()

In [ ]:
df["state"].value_counts()

**Discuss:** some `access` values are missing (NaN). Would a whitespace-split parser have noticed that, or silently shifted the columns?

## 4. Filter: stations in NRW, still active, with long records
Criteria:
1. state is *Nordrhein-Westfalen*,
2. still active: `date_to` equals the most recent end date in the file,
3. record started in 1970 or earlier.

In [ ]:
# YOUR CODE HERE
print(f"{len(df_nrw)} stations, latest end date: {latest:%Y-%m-%d}")
df_nrw[["name", "altitude", "date_from", "date_to"]]

## 5. Georeference: from lat/lon columns to geometries
**Predict:** in which coordinate reference system (CRS) are `lat`/`lon` given? What goes wrong if we forget to declare it?

In [ ]:
# YOUR CODE HERE
gdf_nrw.crs

For NRW we work in **ETRS89 / UTM zone 32N (EPSG:25832)**: metric coordinates, used by all official NRW geodata.
We will look at CRS in depth next session. For now: re-project and compare the coordinate values.

In [ ]:
gdf_nrw_25832 = gdf_nrw.to_crs("EPSG:25832")
gdf_nrw_25832[["name", "geometry"]].head()

## 6. Persist and view
### 6.1 GeoPackage
A GeoPackage is a single SQLite file holding one or more layers: the standard exchange format of this course.

In [ ]:
gpkg = OUT_DIR / "dwd_stations.gpkg"
gdf_all = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326").to_crs("EPSG:25832")
gdf_all.to_file(gpkg, layer="kl_stations_all", driver="GPKG")
gdf_nrw_25832.to_file(gpkg, layer="kl_stations_nrw_active", driver="GPKG")
gpd.list_layers(gpkg)

### 6.2 QGIS: view and check
1. Download `data/derived/dwd_stations.gpkg` from the JupyterHub (right-click → Download) and drag it into QGIS.
2. Set the project CRS to **EPSG:25832**.
3. Add the NRW topographic map as **WMS** (*Layer → Add Layer → Add WMS/WMTS Layer → New*):
   `https://www.wms.nrw.de/geobasis/wms_nw_dtk` (alternative orthophotos: `https://www.wms.nrw.de/geobasis/wms_nw_dop`).
4. Check: are all active stations inside NRW? Does *Aachen* sit on Aachen? Label the points with `station_id` and `name`.

QGIS here is the **viewer and quality check** of what our code produced, and the **OGC client** for the basemap.

### 6.3 Quick web map in the notebook

In [ ]:
gdf_nrw.explore(column="altitude", cmap="viridis", tooltip=["name", "altitude", "date_from"],
                marker_kwds={"radius": 6})

## 7. Exercise (pairs): transfer to another DWD product
The hourly precipitation network (needed in session 10) has its own station list:

`https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/precipitation/recent/RR_Stundenwerte_Beschreibung_Stationen.txt`

1. Download it with `fetch()` into a suitable folder below `data/original/dwd/`.
2. Inspect it. Are the column positions the same? Check, do not assume.
3. Select all **active** precipitation stations in NRW and add them as a third layer `rr_stations_nrw_active` to the GeoPackage.
4. In QGIS: how does the precipitation network compare with the climate network in density?

In [ ]:
# YOUR CODE HERE


## 8. Stretch (optional): from notebook code to a reusable function
Write `read_dwd_station_list(path) -> pd.DataFrame` that encapsulates §3, including the type conversion and a sanity check
(e.g. raise an error if more than 1 % of the coordinates are missing). Write two small `assert`-based tests.
This function will later move into the course package `opengeo.dwd`.

In [ ]:
# YOUR CODE HERE


## 9. Wrap-up
**Exit ticket (2 min):** what was the *muddiest point* today? Write one sentence in the course chat or on a card.

**Homework (until session 2):**
1. Finish §7.
2. Produce a **complete, annotated QGIS map** (EPSG:25832, NRW WMS basemap, labels `station_id – name`, legend, scale bar, title, data source *Deutscher Wetterdienst*) of the active NRW climate and precipitation stations. Export as PNG/PDF.
3. Be ready to present your notebook and map in **5 minutes** at the start of session 2 (2 volunteers).

**Next session:** coordinate reference systems in code: why distances in EPSG:4326 are wrong, and how `pyproj` fixes it.